# 순천향대 졸업요건 RAG 실습

프로젝트 ZIP을 업로드하여 PDF 추출, 단독·다중 문서 검색, 실제 다국어 임베딩/FAISS, 규칙 진단을 실습합니다. 아래 예제는 **가상 테스트 입력**이며 사용자 기록이 아닙니다. 공식 학사시스템의 확인을 대신하지 않습니다.


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, os, sys
uploaded = files.upload()  # graduation-rag-chatbot.zip 선택
name = next(n for n in uploaded if n.lower().endswith('.zip'))
base = Path('/content/graduation_demo').resolve()
base.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
    for info in archive.infolist():
        target = (base / info.filename).resolve()
        if not target.is_relative_to(base):
            raise ValueError('압축 경로가 프로젝트 폴더 밖을 가리킵니다.')
    archive.extractall(base)
root = next(base.rglob('app.py')).parent
os.chdir(root)
sys.path.insert(0, str(root))
print(root)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
# 이미 불러온 라이브러리와 버전 충돌이 있으면 Colab 런타임을 재시작한 뒤 첫 셀부터 다시 실행하세요.


In [ ]:
from src.pdf_loader import load_directory
from src.graduation_rules import load_rules
from src.rag_chain import answer
from src.utils import ROOT
all_docs, errors = load_directory(ROOT / 'data')
original_name = load_rules()['source']
single_docs = [doc for doc in all_docs if doc.metadata['source'] == original_name]
question = 'AI·빅데이터학과 2022학번 졸업인증 총점은?'
text, sources = answer(question, single_docs, 'AI·빅데이터학과', 2022)
print(text)
print('PDF 추출 오류:', errors)


In [ ]:
# 실제 오픈소스 다국어 임베딩 + FAISS (최초 모델 다운로드 필요)
from src.embedding import get_embeddings
from src.text_splitter import split_documents
from src.vector_store import build_store
embeddings = get_embeddings()
store = build_store(split_documents(single_docs), embeddings, ROOT / '.index/colab_single')
text, sources = answer(question, single_docs, 'AI·빅데이터학과', 2022, store=store)
print(text)


In [ ]:
# 선택 사항: 안내받은 추가 PDF가 있을 때만 업로드하세요. 가짜 규정을 생성하지 않습니다.
from src.document_manager import save_pdf
additional = files.upload()
for filename, raw in additional.items():
    if filename.lower().endswith('.pdf'):
        print(save_pdf(raw, filename, ROOT / 'data'))
all_docs, errors = load_directory(ROOT / 'data')
multi_store = build_store(split_documents(all_docs), embeddings, ROOT / '.index/colab_multi')
print(answer(question, all_docs, 'AI·빅데이터학과', 2022, store=multi_store)[0])


In [ ]:
# 학생 입력은 직접 작성하거나 CSV/XLSX를 업로드하세요. 기본값은 빈 기록입니다.
from src.utils import read_course_file
from src.graduation_checker import check
courses = []
activities = {}
evidence = {}
# CSV/XLSX 업로드를 원하는 경우 아래 주석을 해제합니다.
# uploaded_courses = files.upload()
# for filename, raw in uploaded_courses.items():
#     courses += read_course_file(raw, filename)
result = check('AI·빅데이터학과', 2022, courses, activities, complete=False, evidence=evidence)
import pandas as pd
print(result['status'], result['scope'])
display(pd.DataFrame(result['areas']))
display(pd.DataFrame(result['items']))


## 공식 확인 항목 입력

`evidence`의 `중핵교양 국제사회언어영역 택1`에는 `course`, `confirmed`를 넣습니다. 자율활동 세 영역에는 각각 공식 `points`, `confirmed`를 넣습니다. `자격증/경진대회참가/경진대회수상`에는 `certificates` 목록 또는 공식 대회 점수·확인 여부를 넣습니다. 구체적인 입력 구조는 `tests/test_ai_2022.py`의 가상 시나리오를 참고하세요.

선택적으로 `.env`를 만들어 `OPENAI_API_KEY`를 설정하고 `answer(..., use_llm=True)`로 자연어 요약을 사용할 수 있습니다. 키가 없으면 검색 답변이 작동합니다. 이 노트북은 실제 Colab 런타임에서는 검증하지 않았습니다.
